# 07 — Data Splitting / Train-Validation-Test Preparation

This notebook prepares the engineered shipment dataset for machine-learning experiments.

## Required split
- **70% Train**
- **15% Validation**
- **15% Test**

## Critical rule
The split is **time-based**, not random.

The shipment data covers 2024–2025, so observations are sorted chronologically before splitting. This prevents later observations from being used to train models that are evaluated on earlier observations.

### No model training is performed here.
This notebook only creates and validates the train/validation/test partitions.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 1. Project paths

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "shipment_features_engineered.csv"

print("Project root:", PROJECT_ROOT)
print("Input exists:", INPUT_FILE.exists())
print("Split directory:", SPLIT_DIR)


Project root: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Input exists: True
Split directory: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\splits


## 2. Load engineered shipment features

In [3]:
assert INPUT_FILE.exists(), f"Required file not found: {INPUT_FILE}"

df = pd.read_csv(INPUT_FILE, encoding="utf-8")

required_columns = {
    "Date",
    "Disruption_Occurred"
}

missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing required columns: {missing_columns}"

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

assert df["Date"].notna().all(), "Invalid or missing shipment dates found."
assert df["Disruption_Occurred"].isin([0, 1]).all()

print("Shape:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
display(df.head())


Shape: (5000, 51)
Date range: 2024-01-01 to 2025-12-31


,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,Distance_km,Weight_MT,Fuel_Price_Index,Geopolitical_Risk_Score,Weather_Condition,Carrier_Reliability_Score,Lead_Time_Days,Disruption_Occurred,year,month,quarter,day_of_week,day_of_year,week_of_year,month_sin,month_cos,day_of_week_sin,day_of_week_cos,distance_per_mt,fuel_risk_interaction,reliability_risk_inverse,weather_risk_flag,long_lead_time_flag,shipping_year,shipping_month,shipping_baltic_dry_index,shipping_container_rate_usd_40ft,shipping_air_cargo_rate_usd_kg,shipping_bdi_mom_change_pct,shipping_container_yoy_pct,shipping_tanker_rate_aframax_usd_day,shipping_bulk_carrier_handysize_usd_day,shipping_supply_chain_pressure_index,shipping_on_time_delivery_pct,commodity_price,tariff_tariff_rate_pct,tariff_estimated_value_usd_bn,tariff_is_trump_1_0,tariff_is_biden,tariff_is_trump_2_0,tariff_is_retaliation,tariff_is_section_232,tariff_is_section_301,tariff_is_ieepa,disruption_event_flag
0,SC-10000,2025-10-16,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1,2025,10,4,3,289,42,-0.866025,5.000000e-01,0.433884,-0.900969,30.041688,12.150,0.135,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6977.713902,20.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
1,SC-10001,2024-04-24,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1,2024,4,2,2,115,17,0.866025,-5.000000e-01,0.974928,-0.222521,60.214804,17.250,0.408,1,1,2024.0,4.0,2088.0,2870.0,4.20,-1.00,-11.42,32381.0,14640.0,1.44,0.887,5778.355668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,SC-10002,2024-01-26,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0,2024,1,1,4,26,4,0.500000,8.660254e-01,-0.433884,-0.900969,26.002316,9.968,0.327,1,1,2024.0,1.0,1913.0,2500.0,4.22,-3.04,-35.73,28812.0,12058.0,0.82,0.913,5883.525058,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,SC-10003,2024-10-08,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1,2024,10,4,1,282,41,-0.866025,5.000000e-01,0.781831,0.623490,53.794386,2.560,0.168,1,1,2024.0,10.0,1956.0,4340.0,4.38,7.41,57.82,15770.0,12633.0,1.51,0.889,5468.153702,45.3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
4,SC-10004,2024-09-07,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1,2024,9,3,5,251,36,-1.000000,-1.836970e-16,-0.974928,-0.222521,6.350630,5.263,0.259,1,0,2024.0,9.0,1821.0,3900.0,4.24,-9.49,62.50,14710.0,11862.0,1.22,0.839,5581.994881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


## 3. Check duplicate rows and duplicate Shipment IDs

In [4]:
duplicate_rows = int(df.duplicated().sum())

if "Shipment_ID" in df.columns:
    duplicate_ids = int(df["Shipment_ID"].duplicated().sum())
else:
    duplicate_ids = None

print("Duplicate rows:", duplicate_rows)
print("Duplicate Shipment_ID values:", duplicate_ids)

assert duplicate_rows == 0

if duplicate_ids is not None:
    assert duplicate_ids == 0


Duplicate rows: 0
Duplicate Shipment_ID values: 0


## 4. Sort chronologically

This ordering is essential for the time-based split.


In [5]:
df = df.sort_values(
    by=["Date", "Shipment_ID"] if "Shipment_ID" in df.columns else ["Date"]
).reset_index(drop=True)

print("Chronologically sorted:", df["Date"].is_monotonic_increasing)
print("First date:", df["Date"].min().date())
print("Last date:", df["Date"].max().date())

assert df["Date"].is_monotonic_increasing


Chronologically sorted: True
First date: 2024-01-01
Last date: 2025-12-31


## 5. Define the 70 / 15 / 15 split sizes

In [6]:
n = len(df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

print("Total rows:", n)
print("Train rows:", train_end)
print("Validation rows:", validation_end - train_end)
print("Test rows:", n - validation_end)

assert train_end > 0
assert validation_end > train_end
assert validation_end < n


Total rows: 5000
Train rows: 3500
Validation rows: 750
Test rows: 750


## 6. Create chronological partitions

In [7]:
train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

assert len(train_df) + len(validation_df) + len(test_df) == len(df)


Train: (3500, 51)
Validation: (750, 51)
Test: (750, 51)


## 7. Verify date ranges for each partition

In [8]:
def date_summary(name, part):
    return {
        "split": name,
        "rows": len(part),
        "start_date": part["Date"].min(),
        "end_date": part["Date"].max(),
        "disruption_rate": part["Disruption_Occurred"].mean()
    }

date_ranges = pd.DataFrame([
    date_summary("train", train_df),
    date_summary("validation", validation_df),
    date_summary("test", test_df)
])

display(date_ranges)


,split,rows,start_date,end_date,disruption_rate
0,train,3500,2024-01-01,2025-05-30,0.616286
1,validation,750,2025-05-30,2025-09-18,0.604000
2,test,750,2025-09-19,2025-12-31,0.604000


## 8. Verify there is no temporal overlap

In [9]:
assert train_df["Date"].max() <= validation_df["Date"].min()
assert validation_df["Date"].max() <= test_df["Date"].min()

print("Train end <= validation start:", train_df["Date"].max() <= validation_df["Date"].min())
print("Validation end <= test start:", validation_df["Date"].max() <= test_df["Date"].min())
print("Temporal ordering check: PASS")


Train end <= validation start: True
Validation end <= test start: True
Temporal ordering check: PASS


## 9. Inspect target distribution in each split

This is a diagnostic only. No resampling is performed in this notebook.


In [10]:
target_distribution = pd.DataFrame({
    "train": train_df["Disruption_Occurred"].value_counts(normalize=True).sort_index(),
    "validation": validation_df["Disruption_Occurred"].value_counts(normalize=True).sort_index(),
    "test": test_df["Disruption_Occurred"].value_counts(normalize=True).sort_index()
}).fillna(0)

target_distribution.index.name = "Disruption_Occurred"

display(target_distribution)

print("\nTarget counts:")
display(pd.DataFrame({
    "train": train_df["Disruption_Occurred"].value_counts().sort_index(),
    "validation": validation_df["Disruption_Occurred"].value_counts().sort_index(),
    "test": test_df["Disruption_Occurred"].value_counts().sort_index()
}).fillna(0).astype(int))


,train,validation,test
Disruption_Occurred,,,
0,0.383714,0.396,0.396
1,0.616286,0.604,0.604



Target counts:


,train,validation,test
Disruption_Occurred,,,
0,1343,297,297
1,2157,453,453


## 10. Verify no row is present in multiple partitions

In [11]:
if "Shipment_ID" in df.columns:
    train_ids = set(train_df["Shipment_ID"])
    validation_ids = set(validation_df["Shipment_ID"])
    test_ids = set(test_df["Shipment_ID"])

    print("Train ∩ Validation:", len(train_ids & validation_ids))
    print("Train ∩ Test:", len(train_ids & test_ids))
    print("Validation ∩ Test:", len(validation_ids & test_ids))

    assert not (train_ids & validation_ids)
    assert not (train_ids & test_ids)
    assert not (validation_ids & test_ids)
else:
    print("Shipment_ID is not available; row-index partitioning was used.")


Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


## 11. Separate predictors and target

`Shipment_ID` is retained in the saved split files for traceability but is excluded from model predictors.

No scaling, encoding, feature selection, imputation, or model fitting is performed here. Those operations belong to the training/preprocessing stage and must be fitted using the training data only.


In [12]:
TARGET = "Disruption_Occurred"
ID_COLUMNS = ["Shipment_ID"]

def make_xy(part):
    X = part.drop(
        columns=[TARGET] + [c for c in ID_COLUMNS if c in part.columns]
    ).copy()
    y = part[TARGET].astype(int).copy()
    return X, y

X_train, y_train = make_xy(train_df)
X_validation, y_validation = make_xy(validation_df)
X_test, y_test = make_xy(test_df)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (3500, 49)
y_train: (3500,)
X_validation: (750, 49)
y_validation: (750,)
X_test: (750, 49)
y_test: (750,)


## 12. Check predictor columns are identical across splits

In [13]:
assert list(X_train.columns) == list(X_validation.columns)
assert list(X_train.columns) == list(X_test.columns)

print("Predictor columns identical across all splits: PASS")
print("Number of predictors:", len(X_train.columns))


Predictor columns identical across all splits: PASS
Number of predictors: 49


## 13. Check missing and infinite values

This check does not modify the data. Any issue must be handled later using training-only preprocessing.


In [14]:
def check_numeric_quality(part, name):
    numeric = part.select_dtypes(include=np.number)
    missing = int(numeric.isna().sum().sum())
    infinite = int(np.isinf(numeric.to_numpy()).sum())
    print(f"{name}: missing numeric values = {missing}, infinite values = {infinite}")
    return missing, infinite

train_missing, train_inf = check_numeric_quality(X_train, "Train")
val_missing, val_inf = check_numeric_quality(X_validation, "Validation")
test_missing, test_inf = check_numeric_quality(X_test, "Test")


Train: missing numeric values = 28190, infinite values = 0
Validation: missing numeric values = 10986, infinite values = 0
Test: missing numeric values = 8916, infinite values = 0


## 14. Check categorical values without fitting encoders

The next notebook will handle categorical encoding. Here we only inspect the categories present in each split.

A category appearing only in validation/test is not automatically an error. The model preprocessing must use an encoder strategy that can safely handle unseen categories.


In [15]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical predictors:", categorical_cols)

for col in categorical_cols:
    print(f"\n{col}")
    print("Train unique:", X_train[col].nunique(dropna=False))
    print("Validation unique:", X_validation[col].nunique(dropna=False))
    print("Test unique:", X_test[col].nunique(dropna=False))


Categorical predictors: ['Origin_Port', 'Destination_Port', 'Transport_Mode', 'Product_Category', 'Weather_Condition']

Origin_Port
Train unique: 8
Validation unique: 8
Test unique: 8

Destination_Port
Train unique: 9
Validation unique: 9
Test unique: 9

Transport_Mode
Train unique: 4
Validation unique: 4
Test unique: 4

Product_Category
Train unique: 5
Validation unique: 5
Test unique: 5

Weather_Condition
Train unique: 5
Validation unique: 5
Test unique: 5


## 15. Save the three chronological split files

In [16]:
TRAIN_FILE = SPLIT_DIR / "train.csv"
VALIDATION_FILE = SPLIT_DIR / "validation.csv"
TEST_FILE = SPLIT_DIR / "test.csv"

train_df.to_csv(TRAIN_FILE, index=False, encoding="utf-8")
validation_df.to_csv(VALIDATION_FILE, index=False, encoding="utf-8")
test_df.to_csv(TEST_FILE, index=False, encoding="utf-8")

print("Saved:")
print(TRAIN_FILE)
print(VALIDATION_FILE)
print(TEST_FILE)


Saved:
C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\splits\train.csv
C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\splits\validation.csv
C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\splits\test.csv


## 16. Save predictor/target arrays as CSV files

These files are convenient for later modeling notebooks. The original split files remain the authoritative traceable datasets.


In [17]:
X_train_file = SPLIT_DIR / "X_train.csv"
y_train_file = SPLIT_DIR / "y_train.csv"
X_validation_file = SPLIT_DIR / "X_validation.csv"
y_validation_file = SPLIT_DIR / "y_validation.csv"
X_test_file = SPLIT_DIR / "X_test.csv"
y_test_file = SPLIT_DIR / "y_test.csv"

X_train.to_csv(X_train_file, index=False, encoding="utf-8")
y_train.to_csv(y_train_file, index=False, encoding="utf-8")
X_validation.to_csv(X_validation_file, index=False, encoding="utf-8")
y_validation.to_csv(y_validation_file, index=False, encoding="utf-8")
X_test.to_csv(X_test_file, index=False, encoding="utf-8")
y_test.to_csv(y_test_file, index=False, encoding="utf-8")

print("Predictor/target files saved.")


Predictor/target files saved.


## 17. Final validation

In [18]:
validation = {
    "total_rows": len(df),
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "train_fraction": len(train_df) / len(df),
    "validation_fraction": len(validation_df) / len(df),
    "test_fraction": len(test_df) / len(df),
    "chronological_order": (
        train_df["Date"].max() <= validation_df["Date"].min()
        and validation_df["Date"].max() <= test_df["Date"].min()
    ),
    "duplicate_rows": int(df.duplicated().sum()),
    "train_validation_id_overlap": (
        len(set(train_df["Shipment_ID"]) & set(validation_df["Shipment_ID"]))
        if "Shipment_ID" in df.columns else 0
    ),
    "train_test_id_overlap": (
        len(set(train_df["Shipment_ID"]) & set(test_df["Shipment_ID"]))
        if "Shipment_ID" in df.columns else 0
    ),
    "validation_test_id_overlap": (
        len(set(validation_df["Shipment_ID"]) & set(test_df["Shipment_ID"]))
        if "Shipment_ID" in df.columns else 0
    ),
    "train_file_exists": TRAIN_FILE.exists(),
    "validation_file_exists": VALIDATION_FILE.exists(),
    "test_file_exists": TEST_FILE.exists(),
    "X_train_file_exists": X_train_file.exists(),
    "y_train_file_exists": y_train_file.exists(),
    "X_validation_file_exists": X_validation_file.exists(),
    "y_validation_file_exists": y_validation_file.exists(),
    "X_test_file_exists": X_test_file.exists(),
    "y_test_file_exists": y_test_file.exists(),
}

display(pd.DataFrame([validation]))

assert validation["total_rows"] == 5000
assert validation["train_rows"] == 3500
assert validation["validation_rows"] == 750
assert validation["test_rows"] == 750
assert validation["chronological_order"]
assert validation["duplicate_rows"] == 0
assert validation["train_validation_id_overlap"] == 0
assert validation["train_test_id_overlap"] == 0
assert validation["validation_test_id_overlap"] == 0
assert validation["train_file_exists"]
assert validation["validation_file_exists"]
assert validation["test_file_exists"]
assert validation["X_train_file_exists"]
assert validation["y_train_file_exists"]
assert validation["X_validation_file_exists"]
assert validation["y_validation_file_exists"]
assert validation["X_test_file_exists"]
assert validation["y_test_file_exists"]

print("07_data_splitting completed successfully.")


,total_rows,train_rows,validation_rows,test_rows,train_fraction,validation_fraction,test_fraction,chronological_order,duplicate_rows,train_validation_id_overlap,train_test_id_overlap,validation_test_id_overlap,train_file_exists,validation_file_exists,test_file_exists,X_train_file_exists,y_train_file_exists,X_validation_file_exists,y_validation_file_exists,X_test_file_exists,y_test_file_exists
0,5000,3500,750,750,0.7,0.15,0.15,True,0,0,0,0,True,True,True,True,True,True,True,True,True


07_data_splitting completed successfully.


# Important note for Notebook 08

The test set must remain untouched until final model evaluation.

Notebook 08 should:
1. Fit preprocessing on **training data only**.
2. Transform validation and test data using that fitted preprocessing.
3. Remove constant features based on training data only.
4. Encode categorical variables safely.
5. Scale numeric variables where required.
6. Avoid target leakage.

No test-set performance should be used for model selection.
